In [ ]:
"""
XGBoost + Optuna 5-fold CV (R² mean) + iPhone Style Plotting
Sequential processing for AM-I, AM-II datasets

Deterministic / Reproducible version:
- Fix all seeds (python, numpy, optuna, xgboost)
- Force single-thread for XGBoost + CV + learning_curve to avoid nondeterminism
- Avoid data leakage: tuning and learning curves only use TRAIN set; TEST only for final evaluation
"""

# =========================================================
# 0. Reproducibility: environment + seeds (MUST be set early)
# =========================================================
import os
SEED = 42

# Make hashing deterministic (affects dict/set order in some contexts)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Force single-threaded BLAS/OpenMP to avoid nondeterministic reductions
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import random
random.seed(SEED)

import pandas as pd
import numpy as np
np.random.seed(SEED)

import optuna
import joblib
import xgboost as xgb

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import learning_curve


# =========================================================
# 1. Configuration
# =========================================================
DATA_FOLDER = './1-train_test_split'
OUTPUT_ROOT = os.path.join('./', '2-xgb-models')
os.makedirs(OUTPUT_ROOT, exist_ok=True)

FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS   = [f'col{i}'   for i in range(823)]
MG_COLS   = [f'fp_{i}'   for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL   = 'UV_RT-s'

# iPhone Style Color Palette
IPHONE_COLORS = {
    'scatter': '#007AFF',
    'line':    '#AEAEB2',
    'text':    '#000000'
}

# Deterministic setting: do NOT parallelize if you want identical repeated runs
DET_NJOBS = 1   # <- change to >1 only if you accept slight nondeterminism risks


# =========================================================
# 2. Utility Functions
# =========================================================
def evaluate(y_true, y_pred):
    """Calculate evaluation metrics."""
    return {
        'R2':  r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred))
    }

def iphone_style_ax(ax):
    """Apply iPhone-style aesthetics to matplotlib axes."""
    ax.tick_params(axis='both', direction='out', length=6, width=2, labelsize=16)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2)
    ax.grid(False)

def plot_scatter(y_true, y_pred, save_path):
    """Create scatter plot of true vs predicted values with specified formatting."""
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    iphone_style_ax(ax)
    ax.set_aspect('equal', adjustable='box')

    plt.scatter(
        y_true, y_pred,
        alpha=0.8,
        s=70,
        color=IPHONE_COLORS['scatter'],
        edgecolors='none'
    )

    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims, linestyle='--', color=IPHONE_COLORS['line'], linewidth=3)

    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    plt.text(
        0.05, 0.95, f"R² = {r2:.3f}\nMAE = {mae:.2f}",
        transform=ax.transAxes, va='top', fontsize=16, color=IPHONE_COLORS['text']
    )

    plt.xlabel("True RT (s)", fontsize=18, fontweight='bold')
    plt.ylabel("Predicted RT (s)", fontsize=18, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close()

def plot_residuals(y_true, y_pred, save_path):
    """Create residual plot."""
    residuals = y_pred - y_true
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    iphone_style_ax(ax)
    ax.set_aspect('equal', adjustable='box')

    plt.scatter(
        y_pred, residuals,
        alpha=0.8,
        s=70,
        color=IPHONE_COLORS['scatter'],
        edgecolors='none'
    )

    plt.axhline(y=0, linestyle='--', color=IPHONE_COLORS['line'], linewidth=3)

    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    plt.text(
        0.05, 0.95, f"R² = {r2:.3f}\nMAE = {mae:.2f}",
        transform=ax.transAxes, va='top', fontsize=16, color=IPHONE_COLORS['text']
    )

    plt.xlabel("Predicted RT (s)", fontsize=18, fontweight='bold')
    plt.ylabel("Residuals (Predicted - True)", fontsize=18, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close()

def plot_learning_curve(train_sizes, train_scores, val_scores, save_path):
    """Plot learning curve."""
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    iphone_style_ax(ax)

    plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-',
             color=IPHONE_COLORS['scatter'], linewidth=3, label='Train R²')
    plt.plot(train_sizes, np.mean(val_scores, axis=1), 'o-',
             color=IPHONE_COLORS['line'], linewidth=3, label='Val R²')

    plt.xlabel('Training examples', fontsize=18, fontweight='bold')
    plt.ylabel('R²', fontsize=18, fontweight='bold')
    plt.title('Learning Curve (XGBoost)', fontsize=17, fontweight='bold')
    plt.legend(fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close()

def check_data_files(dataset_name):
    """Check if data files exist and their structure."""
    train_file = os.path.join(DATA_FOLDER, f"{dataset_name}_train.csv")
    test_file = os.path.join(DATA_FOLDER, f"{dataset_name}_test.csv")

    print(f"\n🔍 Checking data files for {dataset_name}:")
    print(f"   Train file: {train_file}")
    print(f"   Test file: {test_file}")

    if not os.path.exists(train_file):
        print(f"   ❌ Train file not found!")
        return False
    if not os.path.exists(test_file):
        print(f"   ❌ Test file not found!")
        return False

    try:
        train_df_head = pd.read_csv(train_file, nrows=1)
        test_df_head = pd.read_csv(test_file, nrows=1)

        print(f"   ✅ Files found. Checking structure...")
        print(f"   Train shape: {pd.read_csv(train_file).shape}")
        print(f"   Test shape: {pd.read_csv(test_file).shape}")

        if TARGET_COL not in train_df_head.columns:
            print(f"   ❌ Target column '{TARGET_COL}' not found in train data!")
            print(f"   Available columns: {list(train_df_head.columns[:10])} ...")
            return False

        missing_features = [col for col in ALL_FEATURES[:10] if col not in train_df_head.columns]
        if missing_features:
            print(f"   ⚠️  Some features missing (first 10 checked): {missing_features[:5]} ...")

        _ = test_df_head  # silence linter
        return True

    except Exception as e:
        print(f"   ❌ Error reading files: {str(e)}")
        return False


# =========================================================
# 3. Main Processing Function
# =========================================================
def process_dataset(dataset_name):
    """Process a single dataset with XGBoost and Optuna optimization (deterministic)."""
    train_file = os.path.join(DATA_FOLDER, f"{dataset_name}_train.csv")
    test_file  = os.path.join(DATA_FOLDER, f"{dataset_name}_test.csv")

    if not check_data_files(dataset_name):
        print(f"[SKIP] Cannot process {dataset_name} due to missing or invalid data files")
        return None

    print(f"\n🚀 Processing dataset: {dataset_name}")
    output_dir = os.path.join(OUTPUT_ROOT, dataset_name)
    os.makedirs(output_dir, exist_ok=True)

    # ---------- Load Data ----------
    print(f"   📥 Loading data from {DATA_FOLDER}...")
    train_df = pd.read_csv(train_file)
    test_df  = pd.read_csv(test_file)

    print(f"   📊 Data loaded:")
    print(f"     Train set: {len(train_df)} samples, {len(train_df.columns)} columns")
    print(f"     Test set: {len(test_df)} samples, {len(test_df.columns)} columns")
    print(f"     Target range (train): {train_df[TARGET_COL].min():.2f} - {train_df[TARGET_COL].max():.2f}")
    print(f"     Target range (test): {test_df[TARGET_COL].min():.2f} - {test_df[TARGET_COL].max():.2f}")

    # Features/Target
    X_train = train_df[ALL_FEATURES]
    y_train = train_df[TARGET_COL]
    X_test  = test_df[ALL_FEATURES]
    y_test  = test_df[TARGET_COL]

    # DMatrix (deterministic if data order fixed)
    dtrain_full = xgb.DMatrix(X_train, label=y_train)
    dtest       = xgb.DMatrix(X_test)

    # Precompute var(y_train) once to avoid any hidden differences
    var_y = float(np.var(y_train.values))

    # ---------- Optuna Hyperparameter Optimization ----------
    print(f"   🔍 Optimizing hyperparameters for {dataset_name}...")

    def objective(trial):
        # IMPORTANT: include nthread=1 + seed to make xgb.cv deterministic
        params = {
            'max_depth':        trial.suggest_int('max_depth', 3, 10),
            'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),

            'objective':        'reg:squarederror',
            'tree_method':      'hist',
            'eval_metric':      'rmse',

            # Reproducibility keys
            'seed':             SEED,
            'nthread':          DET_NJOBS,
        }

        cv = xgb.cv(
            params,
            dtrain_full,
            num_boost_round=1000,
            nfold=5,
            early_stopping_rounds=50,
            metrics='rmse',
            seed=SEED,
            verbose_eval=False,
            # Make folds deterministic
            shuffle=True
        )

        rmse = float(cv['test-rmse-mean'].iloc[-1])
        r2 = 1.0 - (rmse ** 2) / var_y
        return r2

    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction='maximize', sampler=sampler)

    # Deterministic: n_jobs=1 (parallel trials can reorder RNG consumption)
    study.optimize(objective, n_trials=30, n_jobs=1)

    best_params = study.best_params
    final_params = {
        'max_depth':        best_params['max_depth'],
        'learning_rate':    best_params['learning_rate'],
        'subsample':        best_params['subsample'],
        'colsample_bytree': best_params['colsample_bytree'],
        'min_child_weight': best_params['min_child_weight'],

        'objective':        'reg:squarederror',
        'tree_method':      'hist',
        'eval_metric':      'rmse',

        # Reproducibility
        'seed':             SEED,
        'nthread':          DET_NJOBS,
    }

    # Save Optuna trial results
    optuna_log_df = pd.DataFrame(
        [(t.number, t.value, t.params) for t in study.trials],
        columns=['trial', 'val_r2_mean', 'params']
    )
    optuna_log_df.to_csv(os.path.join(output_dir, f"{dataset_name}_optuna_log.csv"), index=False)
    print(f"   ✅ Optuna optimization completed. Best R²: {study.best_value:.4f}")

    # ---------- Train Final XGBoost Model ----------
    print(f"   🏋️ Training final model for {dataset_name}...")

    evals_result = {}
    final_model = xgb.train(
        final_params,
        dtrain_full,
        num_boost_round=1000,
        evals=[(dtrain_full, 'train')],
        evals_result=evals_result,
        verbose_eval=False
    )
    joblib.dump(final_model, os.path.join(output_dir, f"{dataset_name}_final_model.pkl"))

    # ---------- Predictions (NO leakage: test only used here) ----------
    print(f"   📊 Making predictions for {dataset_name}...")
    y_pred       = final_model.predict(dtest)
    y_train_pred = final_model.predict(dtrain_full)

    train_metrics = evaluate(y_train, y_train_pred)
    test_metrics  = evaluate(y_test,  y_pred)

    pd.DataFrame({'true': y_test, 'predicted': y_pred}) \
      .to_csv(os.path.join(output_dir, f"{dataset_name}_test_predictions.csv"), index=False)

    pd.DataFrame([train_metrics, test_metrics], index=['train', 'test']) \
      .to_csv(os.path.join(output_dir, f"{dataset_name}_evaluation_summary.csv"))

    # ---------- Generate Plots ----------
    print(f"   🎨 Generating plots for {dataset_name}...")
    plot_scatter(y_test, y_pred, os.path.join(output_dir, f"{dataset_name}_scatter.png"))
    plot_residuals(y_test, y_pred, os.path.join(output_dir, f"{dataset_name}_residuals.png"))

    # Learning Curve (ONLY on train set; deterministic CV)
    # Use sklearn wrapper with fixed random_state and n_jobs=1
    xgb_sklearn = xgb.XGBRegressor(
        max_depth=final_params['max_depth'],
        learning_rate=final_params['learning_rate'],
        subsample=final_params['subsample'],
        colsample_bytree=final_params['colsample_bytree'],
        min_child_weight=final_params['min_child_weight'],
        n_estimators=1000,
        objective='reg:squarederror',
        tree_method='hist',
        random_state=SEED,
        n_jobs=DET_NJOBS,
        verbosity=0
    )

    cv_split = KFold(n_splits=5, shuffle=True, random_state=SEED)

    train_sizes, train_scores, val_scores = learning_curve(
        xgb_sklearn,
        X_train, y_train,
        cv=cv_split,
        scoring='r2',
        n_jobs=DET_NJOBS,
        train_sizes=np.linspace(0.1, 1.0, 5)
    )
    plot_learning_curve(train_sizes, train_scores, val_scores,
                        os.path.join(output_dir, f"{dataset_name}_learning_curve.png"))

    # ---------- Feature Importance ----------
    print(f"   📈 Calculating feature importance for {dataset_name}...")
    importance = final_model.get_score(importance_type='gain')
    importance_df = pd.DataFrame({
        'feature': list(importance.keys()),
        'importance': list(importance.values())
    }).sort_values('importance', ascending=False)
    importance_df.to_csv(os.path.join(output_dir, f"{dataset_name}_feature_importance.csv"), index=False)

    # Plot top 20 features
    top_n = 20
    plot_df = importance_df.head(top_n) if len(importance_df) > top_n else importance_df

    plt.figure(figsize=(10, 8))
    ax = plt.gca()
    iphone_style_ax(ax)

    # NOTE: keep your original rule; here we still use a colormap to create the bar colors
    colors = plt.cm.Blues(np.linspace(0.3, 1, len(plot_df)))
    ax.barh(range(len(plot_df)), plot_df['importance'], color=colors)
    ax.set_yticks(range(len(plot_df)))
    ax.set_yticklabels(plot_df['feature'], fontsize=10)
    ax.set_xlabel('Feature Importance (Gain)', fontsize=18, fontweight='bold')
    ax.set_title(f'Top {len(plot_df)} Feature Importance - {dataset_name}', fontsize=16, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{dataset_name}_feature_importance.png"), dpi=600)
    plt.close()

    print(f"✅ Completed: {dataset_name}")
    print(f"   Test R²: {test_metrics['R2']:.4f}, MAE: {test_metrics['MAE']:.4f}, RMSE: {test_metrics['RMSE']:.4f}")
    print(f"   Results saved to: {output_dir}\n")

    return {
        'dataset': dataset_name,
        'best_r2': float(study.best_value),
        'test_r2': float(test_metrics['R2']),
        'test_mae': float(test_metrics['MAE']),
        'test_rmse': float(test_metrics['RMSE']),
        'train_r2': float(train_metrics['R2']),
        'n_train': int(len(train_df)),
        'n_test': int(len(test_df))
    }


# =========================================================
# 4. Sequential Processing for AM Datasets
# =========================================================
if __name__ == "__main__":
    TARGET_DATASETS = ['AM-I-filtered_with_labels_k4', 'AM-II-filtered_with_labels_k4']

    print("=" * 60)
    print("XGBoost Sequential Processing for AM Datasets (Deterministic)")
    print("=" * 60)
    print(f"📁 Data folder: {os.path.abspath(DATA_FOLDER)}")
    print(f"📁 Output folder: {os.path.abspath(OUTPUT_ROOT)}")
    print(f"🎯 SEED = {SEED}, DET_NJOBS = {DET_NJOBS}")

    if not os.path.exists(DATA_FOLDER):
        print(f"\n❌ ERROR: Data folder '{DATA_FOLDER}' does not exist!")
        print("Please create the folder and place your data files there.")
        exit(1)

    print(f"\n📋 Files in data folder:")
    data_files = os.listdir(DATA_FOLDER)
    csv_files = [f for f in data_files if f.endswith('.csv')]
    for file in sorted(csv_files):
        file_path = os.path.join(DATA_FOLDER, file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)
        print(f"   {file} ({file_size:.1f} MB)")

    all_metrics = []
    for dataset in TARGET_DATASETS:
        print(f"\n{'='*40}")
        try:
            metrics = process_dataset(dataset)
            if metrics:
                all_metrics.append(metrics)
        except Exception as e:
            print(f"❌ Error processing {dataset}: {str(e)}")
            import traceback
            traceback.print_exc()

    if all_metrics:
        print("\n" + "=" * 60)
        print("SUMMARY OF RESULTS")
        print("=" * 60)
        summary_df = pd.DataFrame(all_metrics)
        summary_df = summary_df[['dataset', 'n_train', 'n_test', 'best_r2', 'train_r2', 'test_r2', 'test_mae', 'test_rmse']]
        print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

        summary_path = os.path.join(OUTPUT_ROOT, 'summary_results.csv')
        summary_df.to_csv(summary_path, index=False)
        print(f"\n📋 Summary saved to: {summary_path}")

        # Comparison plot
        plt.figure(figsize=(10, 6))
        ax = plt.gca()
        iphone_style_ax(ax)

        x_pos = np.arange(len(all_metrics))
        width = 0.35

        train_r2 = [m['train_r2'] for m in all_metrics]
        test_r2 = [m['test_r2'] for m in all_metrics]

        ax.bar(x_pos - width/2, train_r2, width, label='Train R²', color=IPHONE_COLORS['scatter'])
        ax.bar(x_pos + width/2, test_r2, width, label='Test R²', color=IPHONE_COLORS['line'])

        ax.set_xlabel('Dataset', fontsize=18, fontweight='bold')
        ax.set_ylabel('R² Score', fontsize=18, fontweight='bold')
        ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels([m['dataset'] for m in all_metrics], fontsize=10)
        ax.legend(fontsize=14)
        plt.tight_layout()

        comp_path = os.path.join(OUTPUT_ROOT, 'performance_comparison.png')
        plt.savefig(comp_path, dpi=600)
        plt.close()
        print(f"📊 Comparison plot saved to: {comp_path}")

    print("\n✨ All processing completed! ✨")


/home/xuxianyan/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


XGBoost Sequential Processing for AM Datasets (Deterministic)
📁 Data folder: /home/xuxianyan/uplc/uplc-260211/1-train_test_split
📁 Output folder: /home/xuxianyan/uplc/uplc-260211/2-xgb-models-test
🎯 SEED = 42, DET_NJOBS = 1

📋 Files in data folder:
   AM-I-filtered_with_labels_k4_test.csv (4.8 MB)
   AM-I-filtered_with_labels_k4_train.csv (42.9 MB)
   AM-II-filtered_with_labels_k4_test.csv (1.3 MB)
   AM-II-filtered_with_labels_k4_train.csv (11.8 MB)


🔍 Checking data files for AM-I-filtered_with_labels_k4:
   Train file: ./1-train_test_split/AM-I-filtered_with_labels_k4_train.csv
   Test file: ./1-train_test_split/AM-I-filtered_with_labels_k4_test.csv
   ✅ Files found. Checking structure...
   Train shape: (6018, 1857)
   Test shape: (670, 1857)

🚀 Processing dataset: AM-I-filtered_with_labels_k4
   📥 Loading data from ./1-train_test_split...
   📊 Data loaded:
     Train set: 6018 samples, 1857 columns
     Test set: 670 samples, 1857 columns
     Target range (train): 33.00 - 112.20


[I 2026-02-12 18:18:36,868] A new study created in memory with name: no-name-d8c15781-0645-4fff-84bd-a747dbb43a36


   🔍 Optimizing hyperparameters for AM-I-filtered_with_labels_k4...


[I 2026-02-12 18:20:08,947] Trial 0 finished with value: 0.8876384554951353 and parameters: {'max_depth': 5, 'learning_rate': 0.22648248189516848, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183, 'min_child_weight': 2}. Best is trial 0 with value: 0.8876384554951353.
[I 2026-02-12 18:21:58,924] Trial 1 finished with value: 0.47141616567192557 and parameters: {'max_depth': 4, 'learning_rate': 0.0013927723945289009, 'subsample': 0.9330880728874675, 'colsample_bytree': 0.8005575058716043, 'min_child_weight': 8}. Best is trial 0 with value: 0.8876384554951353.
[I 2026-02-12 18:23:21,751] Trial 2 finished with value: 0.8863137551953968 and parameters: {'max_depth': 3, 'learning_rate': 0.2526878207508456, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'min_child_weight': 2}. Best is trial 0 with value: 0.8876384554951353.
[I 2026-02-12 18:25:00,994] Trial 3 finished with value: 0.7222570825484413 and parameters: {'max_depth': 4, 'learning_rat

   ✅ Optuna optimization completed. Best R²: 0.8966
   🏋️ Training final model for AM-I-filtered_with_labels_k4...
   📊 Making predictions for AM-I-filtered_with_labels_k4...
   🎨 Generating plots for AM-I-filtered_with_labels_k4...
   📈 Calculating feature importance for AM-I-filtered_with_labels_k4...
✅ Completed: AM-I-filtered_with_labels_k4
   Test R²: 0.8940, MAE: 3.2911, RMSE: 4.3844
   Results saved to: ./2-xgb-models-test/AM-I-filtered_with_labels_k4



🔍 Checking data files for AM-II-filtered_with_labels_k4:
   Train file: ./1-train_test_split/AM-II-filtered_with_labels_k4_train.csv
   Test file: ./1-train_test_split/AM-II-filtered_with_labels_k4_test.csv
   ✅ Files found. Checking structure...
   Train shape: (1650, 1857)
   Test shape: (186, 1857)

🚀 Processing dataset: AM-II-filtered_with_labels_k4
   📥 Loading data from ./1-train_test_split...


[I 2026-02-12 19:16:08,379] A new study created in memory with name: no-name-5de9db67-d6c5-44b2-917c-79a5a60cbf07


   📊 Data loaded:
     Train set: 1650 samples, 1857 columns
     Test set: 186 samples, 1857 columns
     Target range (train): 30.60 - 80.40
     Target range (test): 33.60 - 75.00
   🔍 Optimizing hyperparameters for AM-II-filtered_with_labels_k4...


[I 2026-02-12 19:16:37,352] Trial 0 finished with value: 0.8372526579024919 and parameters: {'max_depth': 5, 'learning_rate': 0.22648248189516848, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183, 'min_child_weight': 2}. Best is trial 0 with value: 0.8372526579024919.
[I 2026-02-12 19:17:34,234] Trial 1 finished with value: 0.5346977243484323 and parameters: {'max_depth': 4, 'learning_rate': 0.0013927723945289009, 'subsample': 0.9330880728874675, 'colsample_bytree': 0.8005575058716043, 'min_child_weight': 8}. Best is trial 0 with value: 0.8372526579024919.
[I 2026-02-12 19:18:07,726] Trial 2 finished with value: 0.8528776296148198 and parameters: {'max_depth': 3, 'learning_rate': 0.2526878207508456, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'min_child_weight': 2}. Best is trial 2 with value: 0.8528776296148198.
[I 2026-02-12 19:19:01,944] Trial 3 finished with value: 0.7591032422814001 and parameters: {'max_depth': 4, 'learning_rate